# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIRˆ² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library. We will:
- Load the Croissant schema and metadata
- List record sets and fields (referencing entities by their `@id`)
- Extract tabular data for analysis
- Perform exploratory data analysis (EDA)
- Visualize and summarize findings

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset (this downloads metadata and discovers record sets/fields)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, "version", ""))
print("Identifier:", getattr(metadata, "identifier", ""))

## 2. Data Overview
Review available record sets, fields (by `@id`), and get a preview of their schema. 

We will list all available record sets, their fields (columns), and reference them by their `@id` for subsequent steps.

In [ ]:
# List the available record sets
print("Available Record Sets (@id and name):")
for rs in dataset.record_sets:
    print(f"  @id: {rs['@id']}  |  name: {rs.get('name', '(no name)')}")

# For each record set, list the fields (columns) and their IDs and types
record_sets_info = {}
for rs in dataset.record_sets:
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    columns = []
    for f in fields:
        fid = f.get('@id')
        fname = f.get('name', '')
        ftype = f.get('dataType', 'unknown')
        columns.append({'@id': fid, 'name': fname, 'dataType': ftype})
    record_sets_info[rs['@id']] = columns

# Print summary
for rs_id, columns in record_sets_info.items():
    print(f"\nRecord Set @id: {rs_id}  Columns:")
    for col in columns:
        print(f"    @id: {col['@id']}  |  name: {col['name'] or '(no name)'}  |  dataType: {col['dataType']}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All data references use `@id` as shown above.

*Note: This dataset typically has a primary record set for clinical/biomarker records. We'll select the first one listed above, but you can change `main_record_set_id` below to any valid `@id` from the output above.*

In [ ]:
record_sets = [rs['@id'] for rs in dataset.record_sets]

# Set your main record set @id (select the primary record set you're interested in)
main_record_set_id = record_sets[0]
print(f"Main record set selected: {main_record_set_id}")

dataframes = {}
for rs_id in record_sets:
    # Load records for each record set @id (if any)
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records) if records else pd.DataFrame()
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set {rs_id}")

# Display columns and first records for the main record set
main_columns = dataframes[main_record_set_id].columns.tolist()
print("\nColumns in main record set:", main_columns)
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

- Select a numeric field by its `@id` (e.g. 'age', diagnosis interval, etc; see cells above for available fields)
- Filter records for demonstration, normalize a numeric column, and group by a key attribute (`group_field`)

Edit `numeric_field_id` and `group_field_id` variables below to use any available columns from your main record set.

In [ ]:
# Example: suppose 'age_at_second_crc' and 'sex' are @ids in the record set (modify as needed)

# Find first numeric field (for demo)
numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    if dataframes[main_record_set_id][col].dtype.kind in 'iufc':
        numeric_field_id = col
        break
if numeric_field_id is None:
    # If no numeric columns auto-detected, pick one manually (below is a common @id pattern for age)
    # numeric_field_id = 'age_at_second_crc' # example, use valid @id from previous cell
    raise ValueError('No numeric field detected. Please manually set `numeric_field_id` to a valid numeric column @id.')
print(f"Numeric field selected: {numeric_field_id}")

threshold = dataframes[main_record_set_id][numeric_field_id].mean()  # Set to mean for demo
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field (standard score)
mean = filtered_df[numeric_field_id].mean()
std = filtered_df[numeric_field_id].std()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field (e.g. 'sex' or 'msi_status'; use an actual @id from earlier)
group_field_id = None
for col in ['sex', 'Sex', 'gender', 'msi_status', 'anatomical_site']:
    if col in dataframes[main_record_set_id].columns:
        group_field_id = col
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped_df)
else:
    print("No suitable grouping field found in the dataset columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to a grouping categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(data=filtered_df, x=numeric_field_id, bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id} (filtered > mean)')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field available, plot boxplot
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(7,5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- We successfully loaded the dataset metadata and extracted tabular data using the `mlcroissant` library, referring to all data entities exclusively by their `@id`.
- Key record sets and fields were reviewed; data was filtered and normalized, and group-wise summaries computed as examples.
- Visualizations help characterize distributions and reveal potential clinical subgroup differences in numeric biomarkers.

*You may now apply further data science analyses, modeling, or domain-specific research based on this FAIR^2 Croissant dataset schema!*